In [9]:
# --- A) Relative-to-baseline scatter from fatigue_budget.json, using scenario FOLDERS ---

import json
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from collections import Counter

# ==== CONFIG ====
marker_size = 10
marker_symbol = "triangle-up"
plot_width = 1200
plot_height = 600
# ================

# First folder is baseline; the rest are scenarios to compare against it
scenario_dirs = [
#     r"/groups/SUDOCO/Task23/sudoco_task2.3/baselines/HKN_randTS_10min",
#     r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_randTS_10min_lookuo_only",
#     r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_randTS_10min_override_bats",
#     r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_randTS_10min_rules_WFFCact_lookup_REV_ENRG_only",
#     r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_randTS_10min_shut_REV_WFFC_wspOnly_override_bats",
#     r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_randTS_10min_rules_shutdownREV_WFFCact_lookup_REV_only",
# r"/groups/SUDOCO/Task23/sudoco_task2.3/baselines/TS_DA_HKN_filled_small_gaps"
# r"/groups/SUDOCO/Task23/sudoco_task2.3/baselines/IEC_IB_baseline_IEA22MW",IEC_IB_baseline_IEA22MW
# r'/groups/SUDOCO/Task23/sudoco_task2.3/baselines/TS_FA_HKN_v3',
# r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_yaw_lookup_Pmax",
# r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_DA_yaw_lookup_Pmax_fixedTI_v2"
# r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_codesign_6725_yaw",
# r"baselines/TS_FA_HKN_v3",
# r"results/HKN_TS_DA_HKN_Revenue_shutdown_500",
# r"results/HKN_TS_DA_HKN_Revenue_shutdown_800",
# r"results/HKN_TS_DA_HKN_Revenue_shutdown_1000",
# r"results/HKN_TS_DA_HKN_Revenue_shutdown_1200",
# r"results/HKN_TS_DA_HKN_Revenue_shutdown_1400",
# r"results/HKN_TS_DA_HKN_Revenue_shutdown_1600",
# r'baselines/IEC_IA_baseline_IEA22MW',
# r'baselines/IEC_IB_baseline_IEA22MW',
# r'baselines/IEC_IC_baseline_IEA22MW',
# r'baselines/IEA_HKN_dist_baseline_22MW_single_clip',
# r'baselines/IEA_HKN_dist_baseline_22MW_single_clipTIboost',
# r'baselines/IEA_HKN_dist_baseline_22MW_single_clipTIboost2',
# r'baselines/IEA_HKN_dist_baseline_22MW_single_clipTIboost3',
# r'baselines/IEA_HKN_dist_baseline_22MW_single_clipTIboost4',
# r'baselines/IEA_HKN_dist_baseline_22MW_single_clipTIboost5',
# r'baselines/Dist_HKN_WSWD_TI_dist_3deg',

r'baselines/IEC_IC_baseline_IEA22MW',
# r"/groups/SUDOCO/Task23/sudoco_task2.3/baselines/Dist_HKN_WSWD_TI_dist_3deg_TIboost",
# r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_dist_yaw_lookup_Pmax_fixedTI_TIboost",
# r'results/HKN_codesign_3147_WSWD_TI_dist_3deg_TIboost',
# r'results/HKN_codesign_6725_WSWD_TI_dist_3deg_TIboost',
# r'results/HKN_codesign_8649_WSWD_TI_dist_3deg_TIboost',
r'results/HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost',


# r"baselines/TS_FA_HKN_v3_TIboost",
#  r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_DA_yaw_lookup_Pmax_fixedTI_v2",
#  r"/groups/SUDOCO/Task23/sudoco_task2.3/results/HKN_DA_yaw_lookup_Pmax_fixedTI_v2_TIboost",

# r'baselines/IEC_IC_baseline_IEA22MW',
# r"baselines/Dist_HKN_WSWD_TI_dist_3deg_TIboost",

# r"baselines/TS_FA_HKN_v4_TIboost",
# r"results/HKN_DA_yaw_lookup_Pmax_fixedTI_v4_TIboost",


]

fatigue_files = [str(Path(d) / "fatigue_budget.json") for d in scenario_dirs]

# Load all JSON files into a DataFrame
data = []
for file in fatigue_files:
    with open(file) as f:
        data.append(json.load(f))
df = pd.DataFrame(data)

# Baseline
baseline = df.iloc[0]

# Relative differences (%) for all numeric columns except 'id'
rel_diff_df = pd.DataFrame()
if "id" in df.columns:
    rel_diff_df["id"] = df["id"]

num_cols = [c for c in df.columns if c != "id"]
for col in num_cols:
    rel_diff_df[col] = 100.0 * (df[col] - baseline[col]) / baseline[col]

# Drop baseline row
rel_diff_df = rel_diff_df.iloc[1:].reset_index(drop=True)

# Reorder so Energy & Revenue are last if present
tail_cols = [c for c in ["Energy", "Revenue"] if c in rel_diff_df.columns]
main_cols = [c for c in rel_diff_df.columns if c not in (["id"] + tail_cols)]
rel_diff_df = rel_diff_df[main_cols + tail_cols]

# Legend labels from scenario folder names
paths = [Path(p) for p in fatigue_files]
parent_counts = Counter([p.parent.name for p in paths])
legend_labels = []
for p in paths[1:]:  # skip baseline
    parent = p.parent.name
    label = parent if parent_counts[parent] == 1 else p.parent.name
    legend_labels.append(label)

# Show the relative differences table in console
print("\nRelative Differences (%) vs baseline (fatigue_budget.json):\n")
print(rel_diff_df)

# Scatter plot
fig = go.Figure()
x_channels = [c for c in rel_diff_df.columns if c != "id"]
for label, (_, row) in zip(legend_labels, rel_diff_df.iterrows()):
    fig.add_trace(go.Scatter(
        x=x_channels,
        y=[row[c] for c in x_channels],
        mode="markers",
        marker=dict(size=marker_size, symbol=marker_symbol),
        name=label,
        showlegend=True
    ))

fig.update_layout(
    title="Relative Percentage Difference to Baseline (fatigue_budget.json)",
    xaxis_title="Channel",
    yaxis_title="Relative Difference (%)",
    yaxis_tickformat=".1f",
    template="plotly_white",
    xaxis=dict(tickangle=-45),
    width=plot_width,
    height=plot_height,
    showlegend=True,
    legend_title_text="Scenario"
)

fig.show()



Relative Differences (%) vs baseline (fatigue_budget.json):

    RA_tbfa    RA_tbss     RA_TBM  RA_TB_torsion    RA_TTM   RA_ttfa  \
0 -18.54373 -21.094461 -13.510983      -8.406064 -8.050867 -8.102104   

     RA_BRM   RA_ebrm   RA_fbrm  RA_blade_torsion  RA_shaft_mx_mb_fixed  \
0 -6.622858 -3.212182  0.263651          7.782543             -7.980868   

   RA_shaft_mz_mb_fixed  RA_gsb_l10  RA_rsb_l10    RA_ADC       Energy  
0             20.497479   -30.37295  -12.277678 -7.754098  3227.201267  


In [10]:
# --- B) Violin plots of per‑turbine relative differences (final cumulative metrics) ---

import pandas as pd
import plotly.express as px
from pathlib import Path

# Reuse scenario_dirs from Cell A. First is baseline.
parquet_files = [str(Path(d) / "turbine_metrics.parquet") for d in scenario_dirs]

def peek_parquet(path, n=5):
    """Optional helper to inspect schema quickly."""
    df = pd.read_parquet(path)
    print(f"\n== {path} ==")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head(n))
    return df

# Uncomment to inspect if needed:
# _ = peek_parquet(parquet_files[0])

# Load baseline and determine columns
df_base = pd.read_parquet(parquet_files[0]).copy()

# Identify turbine ID column (your sample shows 'id')
tid_col = "id" if "id" in df_base.columns else (
    [c for c in df_base.columns if c.lower() in ("turbine", "turbine_id", "tid")][0]
    if any(c.lower() in ("turbine", "turbine_id", "tid") for c in df_base.columns)
    else None
)
if tid_col is None:
    tid_col = "turbine_id"
    df_base[tid_col] = range(len(df_base))

# Channels: all final cumulative columns → everything except the id column
channels = [c for c in df_base.columns if c != tid_col]
if not channels:
    raise ValueError("No data columns found besides the turbine id column.")

# Baseline slice
base = df_base[[tid_col] + channels].set_index(tid_col).sort_index()

# Compute per‑turbine relative differences for each scenario vs baseline
long_records = []
for scen_dir, pq in zip(scenario_dirs[1:], parquet_files[1:]):  # skip baseline
    df_scen = pd.read_parquet(pq).copy()
    if tid_col not in df_scen.columns:
        df_scen[tid_col] = range(len(df_scen))

    scen = df_scen[[tid_col] + channels].set_index(tid_col).reindex(base.index).sort_index()

    # Relative diff (%): 100*(scenario - baseline)/baseline
    denom = base.replace(0, pd.NA)
    rel = 100.0 * (scen - base) / denom

    rel_long = rel.reset_index().melt(id_vars=[tid_col], var_name="Channel", value_name="RelDiff")
    rel_long["Scenario"] = Path(scen_dir).name
    long_records.append(rel_long)

rel_all = pd.concat(long_records, ignore_index=True)

# Summary stats (mean/min/max/count) per Scenario & Channel
summary = (rel_all
           .groupby(["Scenario", "Channel"])["RelDiff"]
           .agg(["mean", "min", "max", "count"])
           .reset_index()
           .sort_values(["Channel", "Scenario"]))
print("\nSummary of per‑turbine relative differences (%) vs baseline (final cumulative metrics):")
print(summary.to_string(index=False))

# Violin plots: one violin per (Channel, Scenario)
fig = px.violin(
    rel_all,
    x="Channel",
    y="RelDiff",
    color="Scenario",
    box=True,          # show inner box (quartiles)
    points="all",      # show all turbine points
)

fig.update_layout(
    title="Per‑Turbine Relative Differences vs Baseline (Final Cumulative Metrics)",
    xaxis_title="Channel",
    yaxis_title="Relative Difference (%)",
    template="plotly_white",
    width=1400,
    height=700,
    legend_title_text="Scenario"
)
fig.update_xaxes(tickangle=-45)
fig.show()



Summary of per‑turbine relative differences (%) vs baseline (final cumulative metrics):
                                   Scenario              Channel       mean        min        max  count
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               Energy  -1.215209  -1.215209  -1.215209      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_ADC -10.506301 -10.506301 -10.506301      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_BRM  -9.055460  -9.055460  -9.055460      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_TBM -34.727379 -34.727379 -34.727379      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost        RA_TB_torsion -15.591337 -15.591337 -15.591337      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_TTM -16.777371 -16.777371 -16.777371      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost     RA_blade_torsion  -2.766742  -2.766742  -2.766742      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost              RA_ebrm  -5.40

In [11]:
# --- B) Violin + per‑turbine colored strip overlay (Energy/Revenue last) ---

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Reuse scenario_dirs from Cell A. First is baseline.
parquet_files = [str(Path(d) / "turbine_metrics.parquet") for d in scenario_dirs]

# Load baseline and determine columns
df_base = pd.read_parquet(parquet_files[0]).copy()

# Identify turbine ID column
tid_col = "id" if "id" in df_base.columns else (
    [c for c in df_base.columns if c.lower() in ("turbine", "turbine_id", "tid")][0]
    if any(c.lower() in ("turbine", "turbine_id", "tid") for c in df_base.columns)
    else None
)
if tid_col is None:
    tid_col = "turbine_id"
    df_base[tid_col] = range(len(df_base))

# Channels: all columns except turbine id
channels = [c for c in df_base.columns if c != tid_col]

# Reorder: put Energy & Revenue last if present
tail = [c for c in ["Energy", "Revenue"] if c in channels]
main = [c for c in channels if c not in tail]
channels_ordered = main + tail

# Baseline slice
base = df_base[[tid_col] + channels_ordered].set_index(tid_col).sort_index()

# Compute per‑turbine relative differences for each scenario vs baseline
long_records = []
for scen_dir, pq in zip(scenario_dirs[1:], parquet_files[1:]):  # skip baseline
    df_scen = pd.read_parquet(pq).copy()
    if tid_col not in df_scen.columns:
        df_scen[tid_col] = range(len(df_scen))

    scen = df_scen[[tid_col] + channels_ordered].set_index(tid_col).reindex(base.index).sort_index()

    denom = base.replace(0, pd.NA)
    rel = 100.0 * (scen - base) / denom

    rel_long = rel.reset_index().melt(id_vars=[tid_col], var_name="Channel", value_name="RelDiff")
    rel_long["Scenario"] = Path(scen_dir).name
    long_records.append(rel_long)

rel_all = pd.concat(long_records, ignore_index=True)

# Summary stats
summary = (rel_all
           .groupby(["Scenario", "Channel"])["RelDiff"]
           .agg(["mean", "min", "max", "count"])
           .reset_index()
           .sort_values(["Channel", "Scenario"]))
print("\nSummary of per‑turbine relative differences (%) vs baseline (final cumulative metrics):")
print(summary.to_string(index=False))

# ---- Figure: violins (by Scenario) + per‑turbine colored strip overlay ----

# Base violin per (Channel, Scenario)
fig = px.violin(
    rel_all,
    x="Channel",
    y="RelDiff",
    color="Scenario",
    box=True,
    points=False,    # points will come from the strip overlay

)
# for trace in fig.data:
#     if trace.type == "violin":
#         # Reduce boxplot width
#         trace.box.width = 0.4  # Default is ~0.5

        # Make the boxplot more transparent
        # trace.box.fillcolor = 'rgba(50, 50, 200, 0.3)'  # RGBA = transparent blue

        # # Optional: line color of box
        # trace.line.color = 'rgba(0, 0, 0, 0.6)'  # Slightly transparent dark gray
        # trace.box.line.color = 'black'

        # # Optional: adjust marker and meanline appearance
        # trace.meanline.color = 'red'
        # trace.meanline.width = 1

        # # Reduce violin fill opacity if desired
        # # trace.opacity = 0.99
        # trace.line.width = 0            # no violin border
        # trace.fillcolor = 'rgba(0,0,0,0)'  # fully transparent fill
        # trace.opacity = 0               # ensure violin body is hidden

# Lock x order so Energy/Revenue appear last
fig.update_xaxes(categoryorder="array", categoryarray=channels_ordered)

fig.update_layout(
    title="Per‑Turbine Relative Differences vs Baseline (Final Cumulative Metrics)",
    xaxis_title="Channel",
    yaxis_title="Relative Difference (%)",
    template="plotly_white",
    width=1400,
    height=700,
    legend_title_text="Scenario"
)
fig.update_xaxes(tickangle=-45)

# ---- Build a consistent color map for turbines across scenarios ----
unique_turbines = pd.unique(rel_all[tid_col])
# Use a long qualitative palette and cycle if needed
palette = (
    px.colors.qualitative.Alphabet
    + px.colors.qualitative.Dark24
    + px.colors.qualitative.Light24
    + px.colors.qualitative.Set3
)
if len(unique_turbines) > len(palette):
    # repeat the palette to cover all turbines
    times = int(np.ceil(len(unique_turbines) / len(palette)))
    palette = (palette * times)[:len(unique_turbines)]

tid_to_color = {tid: palette[i] for i, tid in enumerate(unique_turbines)}

# ---- Overlay a jittered strip of points colored by turbine id ----
# We do this per scenario so points align with each scenario's violin
for scen in sorted(rel_all["Scenario"].unique()):
    df_s = rel_all[rel_all["Scenario"] == scen].copy()

    # Strip with jitter, colored by turbine id; no legend to avoid clutter
    strip = px.strip(
        df_s,
        x="Channel",
        y="RelDiff",
        color=tid_col,
        stripmode="overlay",
        hover_data=[tid_col, "Scenario", "Channel", "RelDiff"],
        color_discrete_map=tid_to_color
    )

    # Tweak aesthetics and hide legends for turbine colors
    for tr in strip.data:
        tr.showlegend = False
        tr.marker.update(size=6, opacity=0.65, line=dict(width=0))
        # Keep traces grouped visually above the corresponding scenario violins
        tr.legendgroup = f"turbines_{scen}"
        fig.add_trace(tr)

fig.show()



Summary of per‑turbine relative differences (%) vs baseline (final cumulative metrics):
                                   Scenario              Channel       mean        min        max  count
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               Energy  -1.215209  -1.215209  -1.215209      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_ADC -10.506301 -10.506301 -10.506301      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_BRM  -9.055460  -9.055460  -9.055460      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_TBM -34.727379 -34.727379 -34.727379      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost        RA_TB_torsion -15.591337 -15.591337 -15.591337      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost               RA_TTM -16.777371 -16.777371 -16.777371      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost     RA_blade_torsion  -2.766742  -2.766742  -2.766742      1
HKN_codesign_6290_WSWD_TI_dist_3deg_TIboost              RA_ebrm  -5.40

In [12]:
import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

# Define spacing between grouped boxes
group_spacing = 0.10
scenarios = sorted(rel_all["Scenario"].unique())
channels = channels_ordered

# Mapping x positions for each (channel, scenario) pair
x_positions = {}
x_base = 0
for channel in channels:
    for scen in scenarios:
        x_positions[(channel, scen)] = x_base
        x_base += 1
    x_base += group_spacing

# Store clean tick labels (just channel name once per group)
tickvals = []
ticktext = []
custom_labels = [
    "TB fore-aft", "TB side-side", "TB proj", "TB_torsion",
    "TT proj", "TT fore-aft", "BR proj",
    "BR edge", "BR flap", "BR torsion",
    "Shaft lat mom", "Shaft torsion", "GS Beating L10",
    "RS Beating L10", "Pitch ADC", "Yaw Travel", "Energy", "Revenue"
]
# for channel in channels:
#     scen0 = scenarios[0]
#     tickvals.append(x_positions[(channel, scen0)] + (len(scenarios)-1)/2)  # center of group
#     ticktext.append(channel)

for i, channel in enumerate(channels):
    scen0 = scenarios[0]
    group_center = x_positions[(channel, scen0)] + (len(scenarios) - 1) / 2
    tickvals.append(group_center)
    ticktext.append(custom_labels[i])


# Add box plots
for channel in channels:
    for scen in scenarios:
        subset = rel_all[(rel_all["Channel"] == channel) & (rel_all["Scenario"] == scen)]
        xpos = x_positions[(channel, scen)]

        fig.add_trace(go.Box(
            y=subset["RelDiff"],
            x=[xpos] * len(subset),
            name=scen,
            boxpoints=False,
            marker_color="black",
            line=dict(width=1),
            fillcolor="rgba(0,0,0,0.1)",
            whiskerwidth=0.8,
            showlegend=False,
            boxmean=True
        ))

# Add scatter points offset slightly to the right of each box
offset = 0.5  # controls how far right the dots appear
for channel in channels:
    for scen in scenarios:
        subset = rel_all[(rel_all["Channel"] == channel) & (rel_all["Scenario"] == scen)].copy()
        xpos = x_positions[(channel, scen)]

        # Offset all points slightly to the right of the box
        subset["x_jitter"] = xpos + offset

        fig.add_trace(go.Scatter(
            x=subset["x_jitter"],
            y=subset["RelDiff"],
            mode="markers",
            marker=dict(
                size=6,
                opacity=0.65,
                color=[tid_to_color[tid] for tid in subset[tid_col]],
                line=dict(width=0)
            ),
            hoverinfo="text",
            text=[f"Turbine {tid}<br>RelDiff: {rd:.2f}%" for tid, rd in zip(subset[tid_col], subset["RelDiff"])],
            showlegend=False
        ))

# Final layout tweaks
fig.update_layout(
    # title="Per-metric relative Differences vs Baseline",
    yaxis_title="Relative Difference (%)",
    xaxis=dict(
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=-45
    ),
    template="plotly_white",
    width=1400,
    height=700,
    yaxis=dict(
        title_font=dict(size=18),   # Y-axis title font size
        tickfont=dict(size=16)      # Y-axis tick labels font size
    ),
    font=dict(
        size=16                    # Global font size (e.g., legend, annotations)
    ),
)


fig.show()
